# MC19 FOV / ROI Viewer

This notebook is a focused copy of the FOV and ROI viewing sections from `~/2p_imaging/2AFC/Test_pilot/test.ipynb`, wired to processed MC19 sessions instead of MC11 sessions.


## Setup


In [ ]:
import os
import sys
from pathlib import Path

import h5py
import numpy as np

MPLCONFIGDIR = Path.home() / 'scratch' / 'tmp' / 'matplotlib'
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('MPLCONFIGDIR', str(MPLCONFIGDIR))

import matplotlib.pyplot as plt

# Reuse the 2AFC notebook plotting/widgets without copying the whole analysis notebook.
TWO_AFC_ROOT = Path.home() / '2p_imaging' / '2AFC'
TWO_AFC_TEST_PILOT = TWO_AFC_ROOT / 'Test_pilot'
PASSIVE_CLOSED_LOOP_ROOT = Path.home() / '2p_imaging' / 'passive_closed_loop'
NOTEBOOK_OUTPUT_ROOT = PASSIVE_CLOSED_LOOP_ROOT / 'Outputs' / 'MC19_FOV_ROI'

for path_entry in [TWO_AFC_ROOT, TWO_AFC_TEST_PILOT]:
    path_str = str(path_entry)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from notebook_tools.io import (
    create_superimposed_mask_images,
    plot_fov_summary,
)

plt.rcParams['axes.grid'] = False


## MC19 Session Paths


In [ ]:
MC19_PROCESSED_ROOTS = [
    Path('/storage/project/r-fnajafi3-0/shared/2P_Imaging/MC19_Processed'),
    Path('/storage/project/r-fnajafi3-0/shared/2P_Imaging/MC19_Processed_New'),
    Path('/storage/cedar/cedar0/cedarp-fnajafi3-0/2p_imaging/MC19_Processed'),
    Path('/storage/scratch1/3/grubin6/2p_processing_results'),
]


def resolve_ops_path(session_dir):
    session_dir = Path(session_dir)
    candidates = [
        session_dir / 'ops.npy',
        session_dir / 'suite2p' / 'plane0' / 'ops.npy',
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def discover_mc19_processed_sessions(roots):
    sessions = []
    seen = set()
    for root in roots:
        if not root.exists():
            continue
        for ops_path in root.rglob('ops.npy'):
            session_dir = ops_path.parent
            if session_dir.name == 'plane0' and session_dir.parent.name == 'suite2p':
                session_dir = session_dir.parent.parent
            if 'MC19' not in session_dir.name and 'MC19' not in str(session_dir):
                continue
            resolved = session_dir.resolve()
            if resolved in seen:
                continue
            seen.add(resolved)
            sessions.append(session_dir)
    return sorted(sessions, key=lambda path: str(path))


list_session_data_path = discover_mc19_processed_sessions(MC19_PROCESSED_ROOTS)
list_session_names = [path.name for path in list_session_data_path]
list_session_output_path = [NOTEBOOK_OUTPUT_ROOT / name for name in list_session_names]
for output_path in list_session_output_path:
    output_path.mkdir(parents=True, exist_ok=True)

print(f'Found {len(list_session_data_path)} processed MC19 sessions with ops.npy:')
for session_name, session_path in zip(list_session_names, list_session_data_path):
    print(f' - {session_name}: {session_path}')


## MC19-Compatible Readers

The original 2AFC notebook reads MC11-style `masks.h5` and root-level `ops.npy`. These readers keep the same notebook variable names and return shapes, while also supporting MC19 Suite2p outputs under `suite2p/plane0`.


In [ ]:
def read_ops(list_session_data_path):
    list_ops = []
    for session_data_path in list_session_data_path:
        session_data_path = Path(session_data_path)
        ops_path = resolve_ops_path(session_data_path)
        if ops_path is None:
            raise FileNotFoundError(f'Could not find ops.npy for {session_data_path}')
        ops = np.load(ops_path, allow_pickle=True).item()
        ops['save_path0'] = str(session_data_path)
        ops['_ops_path'] = str(ops_path)
        list_ops.append(ops)
    return list_ops


def _suite2p_plane_dir(session_dir):
    return Path(session_dir) / 'suite2p' / 'plane0'


def _build_labeled_masks_from_stat(stat_path, shape):
    stat = np.load(stat_path, allow_pickle=True)
    masks = np.zeros(shape, dtype=np.int32)
    for roi_id, roi in enumerate(stat, start=1):
        ypix = np.asarray(roi.get('ypix', []), dtype=np.intp)
        xpix = np.asarray(roi.get('xpix', []), dtype=np.intp)
        valid = (ypix >= 0) & (ypix < shape[0]) & (xpix >= 0) & (xpix < shape[1])
        masks[ypix[valid], xpix[valid]] = roi_id
    return masks


def _labels_for_masks(session_dir, n_rois):
    # MC19 processed folders do not currently carry the MC11 excitatory/inhibitory labels.
    # Use 0 (unsure) for every ROI so the copied 2AFC overlay widget still draws ROI boundaries.
    labels = np.zeros(n_rois + 1, dtype=np.int8)
    return labels


def read_masks(ops):
    session_dir = Path(ops['save_path0'])
    masks_h5 = session_dir / 'masks.h5'
    if masks_h5.exists():
        with h5py.File(masks_h5, 'r') as f:
            labels = np.asarray(f['labels'], dtype=np.int8)
            masks = np.asarray(f['masks_func'], dtype=np.float32)
            mean_func = np.asarray(f['mean_func'], dtype=np.float32)
            max_func = np.asarray(f['max_func'], dtype=np.float32)
            mean_anat = np.asarray(f['mean_anat'], dtype=np.float32) if ops.get('nchannels', 1) == 2 and 'mean_anat' in f else None
            masks_anat = np.asarray(f['masks_anat'], dtype=np.float32) if ops.get('nchannels', 1) == 2 and 'masks_anat' in f else None
        return labels, masks, mean_func, max_func, mean_anat, masks_anat

    mean_func = np.asarray(ops.get('meanImg'), dtype=np.float32)
    if mean_func.size == 0:
        raise KeyError(f'ops for {session_dir} does not contain meanImg')
    max_func = np.asarray(ops.get('max_proj', ops.get('max_proj_chan2', mean_func)), dtype=np.float32)
    mean_anat = None
    if ops.get('nchannels', 1) == 2 and 'meanImg_chan2' in ops:
        mean_anat = np.asarray(ops['meanImg_chan2'], dtype=np.float32)

    # Prefer QC-filtered ROIs when present. Raw Suite2p stat.npy includes every
    # candidate ROI and can greatly overstate accepted ROI counts.
    stat_candidates = [
        session_dir / 'qc_results' / 'stat.npy',
        _suite2p_plane_dir(session_dir) / 'stat.npy',
        session_dir / 'stat.npy',
    ]
    stat_path = next((path for path in stat_candidates if path.exists()), None)
    if stat_path is None:
        masks = np.zeros(mean_func.shape, dtype=np.int32)
        labels = np.zeros(1, dtype=np.int8)
    else:
        masks = _build_labeled_masks_from_stat(stat_path, mean_func.shape)
        labels = _labels_for_masks(session_dir, int(masks.max()))
    masks_anat = None
    return labels, masks.astype(np.float32), mean_func, max_func, mean_anat, masks_anat


def read_dff(ops):
    session_dir = Path(ops['save_path0'])
    candidates = [
        session_dir / 'dff.h5',
        session_dir / 'qc_results' / 'dff.h5',
    ]
    dff_path = next((path for path in candidates if path.exists()), None)
    if dff_path is None:
        raise FileNotFoundError(f'Could not find dff.h5 for {session_dir}')
    with h5py.File(dff_path, 'r') as f:
        return np.asarray(f['dff'], dtype=np.float32)


list_ops = read_ops(list_session_data_path)
print('Loaded ops:', len(list_ops))


## FOV Visualization

This is the copied FOV section from the 2AFC notebook, now using the MC19 session list above. The arrow widget moves through MC19 sessions one figure at a time.


In [ ]:
# Fast, lazy FOV rendering. Do not precompute all sessions; large MC19 sessions have thousands
# of accepted ROIs, and the original 2AFC helper loops over ROI IDs one by one.
from skimage.segmentation import find_boundaries

_fov_cache = {}


def _normalize_channel(image, low=1, high=99):
    image = np.asarray(image, dtype=np.float32)
    vmin, vmax = np.percentile(image, [low, high])
    return np.clip((image - vmin) / (vmax - vmin + 1e-9), 0, 1)


def _colorize(image, channel='green'):
    norm = _normalize_channel(image)
    rgb = np.zeros((*norm.shape, 3), dtype=np.float32)
    if channel == 'red':
        rgb[..., 0] = norm
    else:
        rgb[..., 1] = norm
    return rgb


def _fast_boundary_overlay(base_image, masks, color=(1, 1, 1)):
    base = np.asarray(base_image, dtype=np.float32).copy()
    mask_img = np.rint(np.asarray(masks).squeeze()).astype(np.int32)
    boundaries = find_boundaries(mask_img, mode='outer')
    base[boundaries] = color
    return np.clip(base, 0, 1)


def load_fov_session(index):
    index = int(index)
    if index not in _fov_cache:
        session_name = list_session_names[index]
        ops = list_ops[index]
        labels_i, masks_i, mean_func_i, max_func_i, mean_anat_i, masks_anat_i = read_masks(ops)
        mean_fun_i = _colorize(mean_func_i, channel='green')
        max_fun_i = _colorize(max_func_i, channel='green')
        mean_anat_rgb_i = _colorize(mean_anat_i, channel='red') if mean_anat_i is not None else None
        super_func_i = _fast_boundary_overlay(mean_fun_i, masks_i, color=(1, 1, 1))
        super_anat_i = _fast_boundary_overlay(mean_anat_rgb_i, masks_i, color=(1, 1, 1)) if mean_anat_rgb_i is not None else None
        roi_count_i = int(np.max(np.asarray(masks_i).squeeze()))
        _fov_cache[index] = {
            'session_name': session_name,
            'labels': labels_i,
            'masks': masks_i,
            'mean_func': mean_func_i,
            'max_func': max_func_i,
            'mean_anat': mean_anat_i,
            'mean_fun_rgb': mean_fun_i,
            'max_fun_rgb': max_fun_i,
            'mean_anat_rgb': mean_anat_rgb_i,
            'super_func': super_func_i,
            'super_anat': super_anat_i,
            'roi_count': roi_count_i,
        }
    return _fov_cache[index]


print('Lazy FOV loader ready. Sessions:')
for idx, session_name in enumerate(list_session_names):
    print(f' {idx}: {session_name}')


In [ ]:
# Interactive per-session FOV viewer (lazy; one session is loaded/rendered at a time).
import io
import ipywidgets as widgets
from IPython.display import display

session_index = widgets.IntSlider(value=0, min=0, max=len(list_session_names) - 1, step=1, description='Session')
prev_button = widgets.Button(description='Prev')
next_button = widgets.Button(description='Next')
title_html = widgets.HTML()
image_widget = widgets.Image(format='png')
status_html = widgets.HTML()


def render_fov(index):
    status_html.value = 'Loading...'
    data = load_fov_session(index)
    title_html.value = f"<b>{data['session_name']}</b> | {data['roi_count']} QC/accepted ROIs | index {index}"

    fig, axes = plt.subplots(2, 3, figsize=(16, 10), constrained_layout=True)
    for ax in axes.ravel():
        ax.axis('off')

    axes[0, 0].imshow(data['mean_fun_rgb'])
    axes[0, 0].set_title('Mean Functional')
    axes[0, 1].imshow(data['max_fun_rgb'])
    axes[0, 1].set_title('Max Functional')
    axes[0, 2].imshow(np.asarray(data['masks']).squeeze(), cmap='gray')
    axes[0, 2].set_title('ROI Masks')
    axes[1, 0].imshow(data['super_func'])
    axes[1, 0].set_title('ROI Boundaries on Functional')

    if data['mean_anat_rgb'] is not None:
        axes[1, 1].imshow(data['mean_anat_rgb'])
        axes[1, 1].set_title('Mean Anatomical')
    if data['super_anat'] is not None:
        axes[1, 2].imshow(data['super_anat'])
        axes[1, 2].set_title('ROI Boundaries on Anatomical')

    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=110, bbox_inches='tight')
    plt.close(fig)
    image_widget.value = buf.getvalue()
    status_html.value = ''


def on_slider(change):
    render_fov(change['new'])


def go_prev(_):
    session_index.value = max(session_index.min, session_index.value - 1)


def go_next(_):
    session_index.value = min(session_index.max, session_index.value + 1)


session_index.observe(on_slider, names='value')
prev_button.on_click(go_prev)
next_button.on_click(go_next)

display(widgets.VBox([
    title_html,
    widgets.HBox([prev_button, next_button, session_index]),
    status_html,
    image_widget,
]))
render_fov(session_index.value)


## DFF-Capable Sessions

The DFF widgets are included from the 2AFC notebook. They are limited to MC19 sessions that already have `dff.h5`.


In [ ]:
dff_session_indices = []
for idx, ops in enumerate(list_ops):
    session_dir = Path(ops['save_path0'])
    if (session_dir / 'dff.h5').exists() or (session_dir / 'qc_results' / 'dff.h5').exists():
        dff_session_indices.append(idx)

list_dff_session_names = [list_session_names[idx] for idx in dff_session_indices]
list_dff_ops = [list_ops[idx] for idx in dff_session_indices]

print(f'Found {len(list_dff_session_names)} MC19 sessions with dff.h5:')
for session_name in list_dff_session_names:
    print(f' - {session_name}')


### Raw DFF Trace Quality Viewer

Interactively inspect raw DFF traces by session and neuron.


In [ ]:
import importlib
import notebook_tools.plot_viewers as plot_viewers
importlib.reload(plot_viewers)
from notebook_tools.plot_viewers import show_raw_dff_trace_viewer

if list_dff_session_names:
    show_raw_dff_trace_viewer(
        list_dff_session_names,
        list_dff_ops,
        read_dff,
        max_points=10000,
        max_all_neurons=80,
    )
else:
    print('No MC19 sessions with dff.h5 were found; skipping DFF trace viewer.')


### Click FOV ROI to Inspect DFF

Click an ROI centroid on the FOV to display that neuron trace below.


In [ ]:
# %matplotlib widget
import importlib
import notebook_tools.plot_viewers as plot_viewers
importlib.reload(plot_viewers)

if list_dff_session_names:
    plot_viewers.show_fov_dff_click_viewer(
        list_dff_session_names,
        list_dff_ops,
        read_masks,
        read_dff,
        max_points=10000,
    )
else:
    print('No MC19 sessions with dff.h5 were found; skipping FOV/DFF click viewer.')
